In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import h5py

H5_KEY = 'dataset'


In [ ]:
def process_h5_file(signal_name):

    h5_filename = f"{signal_name}_raw_data.h5"
    source_path = MIT_DATASET_PATH / h5_filename
    
    if not source_path.exists():
        print(f"SKIPPING: {signal_name} (File not found at {source_path})")
        return

    print(f"--- Processing {signal_name} ---")
    
    with h5py.File(source_path, 'r') as f:
        if H5_KEY not in f:
             print(f"ERROR: Key '{H5_KEY}' not found in {h5_filename} available keys: {list(f.keys())}")
             return
             
        # Load the ENTIRE matrix into memory
        # Shape is (Rows, Samples) e.g., (100, 43560) or (530, 230000)
        raw_data = f[H5_KEY][:]
        print(f"Loaded shape: {raw_data.shape}")
        
        # Save as a .npy file
        save_path = get_unet_path(STAGE_RAW, signal=signal_name)
        np.save(save_path, raw_data)
        
    print(f"Saved to {save_path}")

In [ ]:
# Run the processing for all configured datasets
for h5_file in SIGNALS:
    process_h5_file(h5_file)